# 🐦 Conteo de Aves — Formato Pivotado

Salida: una fila por **Foto + Clase**, columnas por cada **Modelo_Confianza**.

In [ ]:
# ============================================================
# 1. CONFIGURA TUS RUTAS
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np

RUTA_FOTOS   = Path('/content/drive/MyDrive/conteo de aves marinas/fotos')
RUTA_MODELOS = Path('/content/drive/MyDrive/conteo de aves marinas/modelos')
RUTA_SALIDA  = Path('/content/drive/MyDrive/conteo de aves marinas/conteos_sahi_pivotado.xlsx')
#TAMANOS      = ['n', 's', 'm', 'l', 'x']
TAMANOS      = ['x']
CONFIDENCIAS = np.arange(0.20, 0.91, 0.01).round(2).tolist()
#CONFIDENCIAS = [0.3,0.56,0.78]

# Parámetros de SAHI
SAHI_SLICE_H = 1280
SAHI_SLICE_W = 1280
SAHI_OVERLAP = 0.2

print('Fotos  :', RUTA_FOTOS,   '✅' if RUTA_FOTOS.exists() else '❌')
print('Modelos:', RUTA_MODELOS, '✅' if RUTA_MODELOS.exists() else '❌')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fotos  : /content/drive/MyDrive/conteo de aves marinas/fotos ✅
Modelos: /content/drive/MyDrive/conteo de aves marinas/modelos ✅


In [ ]:
CONFIDENCIAS

[0.2,
 0.21,
 0.22,
 0.23,
 0.24,
 0.25,
 0.26,
 0.27,
 0.28,
 0.29,
 0.3,
 0.31,
 0.32,
 0.33,
 0.34,
 0.35,
 0.36,
 0.37,
 0.38,
 0.39,
 0.4,
 0.41,
 0.42,
 0.43,
 0.44,
 0.45,
 0.46,
 0.47,
 0.48,
 0.49,
 0.5,
 0.51,
 0.52,
 0.53,
 0.54,
 0.55,
 0.56,
 0.57,
 0.58,
 0.59,
 0.6,
 0.61,
 0.62,
 0.63,
 0.64,
 0.65,
 0.66,
 0.67,
 0.68,
 0.69,
 0.7,
 0.71,
 0.72,
 0.73,
 0.74,
 0.75,
 0.76,
 0.77,
 0.78,
 0.79,
 0.8,
 0.81,
 0.82,
 0.83,
 0.84,
 0.85,
 0.86,
 0.87,
 0.88,
 0.89,
 0.9]

In [ ]:
# ============================================================
# 2. INSTALAR Y CARGAR
# ============================================================


!pip install ultralytics sahi openpyxl -q

from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from collections import defaultdict
import pandas as pd
from pathlib import Path
import yaml

with open(RUTA_MODELOS / 'data.yaml', 'r') as f:
    data = yaml.safe_load(f)
names = data['names']
CLASES = names if isinstance(names, list) else [names[i] for i in sorted(names, key=int)]
print('Clases:', CLASES)



Clases: ['chuita', 'chuita adulta', 'cushuri adulto', 'cushuri juvenil', 'gallinazo cabeza roja', 'gaviota peruana adulta', 'guanay adulto', 'pelicano adulto', 'pelicano juvenil', 'pichon pinguino', 'pichon piquero', 'pinguino adulto', 'pinguino juvenil', 'piquero adulto', 'piquero juvenil', 'zarcillo']


In [ ]:
# ============================================================
# 3. LISTAR FOTOS
# ============================================================

EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
fotos_raw = [f for f in RUTA_FOTOS.iterdir() if f.suffix in EXTS]

# 🔴 CORREGIDO: verificar que el archivo existe realmente antes de incluirlo
fotos = sorted([f for f in fotos_raw if f.exists()])
faltantes = len(fotos_raw) - len(fotos)
if faltantes > 0:
    print(f'⚠️  {faltantes} archivo(s) listado(s) pero no encontrado(s) físicamente (Drive no sincronizado)')

print(f'Total fotos válidas: {len(fotos)}')
for f in fotos[:5]:
    print(' ', f.name)
if len(fotos) > 5:
    print(f'  ... y {len(fotos)-5} más')


Total fotos válidas: 115
  371.jpg
  378.jpg
  380.jpg
  384.jpg
  389.jpg
  ... y 110 más


In [ ]:
# ============================================================
# 4. PREDECIR Y ACUMULAR EN FORMATO LONG (imagen por imagen)
# ============================================================

filas_long = []

for tam in TAMANOS:
    ruta_modelo = RUTA_MODELOS / f'26{tam}.pt'
    if not ruta_modelo.exists():
        print(f'⚠️  No existe: {ruta_modelo}')
        continue

    print(f'\n🔄 Procesando tamaño 26{tam}...')

    for conf in CONFIDENCIAS:
        col_name = f'{tam}_{conf:.2f}'
        print(f'   {col_name} ...', end=' ', flush=True)

        # Cargar modelo en SAHI
        detection_model = AutoDetectionModel.from_pretrained(
            model_type="ultralytics",
            model_path=str(ruta_modelo),
            confidence_threshold=conf,
            device="cuda:0",
        )

        ok_count = 0
        fail_count = 0

        for ruta_foto in fotos:
            foto = ruta_foto.stem

            # 🔴 CORREGIDO: try/except para saltar fotos que fallen
            try:
                result = get_sliced_prediction(
                    str(ruta_foto),
                    detection_model,
                    slice_height=SAHI_SLICE_H,
                    slice_width=SAHI_SLICE_W,
                    overlap_height_ratio=SAHI_OVERLAP,
                    overlap_width_ratio=SAHI_OVERLAP,
                )

                conteo = defaultdict(int)
                for pred in result.object_prediction_list:
                    cls_idx = int(pred.category.id)
                    cls_name = CLASES[cls_idx]
                    conteo[cls_name] += 1

                ok_count += 1

            except Exception as e:
                fail_count += 1
                # Solo imprime el primer error por modelo/conf para no saturar
                if fail_count == 1:
                    print(f'\n      ⚠️  Error en {foto}: {type(e).__name__}')
                continue

            for clase in CLASES:
                filas_long.append({
                    'FOTO': foto,
                    'Clase': clase,
                    'modelo_conf': col_name,
                    'Conteo': conteo.get(clase, 0)
                })

        print(f'OK ({ok_count} fotos, {fail_count} fallos)')

print(f'\n✅ Total filas long: {len(filas_long)}')



🔄 Procesando tamaño 26x...
   x_0.20 ... Performing prediction on 16 slices.
Performing prediction on 12 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 16 slices.
Performing prediction on 20 slices.
Performing prediction on 16 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 16 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction on 12 slices.
Performing prediction on 20 slices.
Performing prediction on 20 slices.
Performing prediction 

In [ ]:
# ============================================================
# 5. PIVOTAR AL FORMATO DESEADO
# ============================================================

df_long = pd.DataFrame(filas_long)

# Pivot: filas = FOTO+Clase, columnas = modelo_conf, valores = Conteo
df_pivot = df_long.pivot_table(
    index=['FOTO', 'Clase'],
    columns='modelo_conf',
    values='Conteo',
    aggfunc='first',
    fill_value=0
).reset_index()

# Aplanar multi-index de columnas si quedó
df_pivot.columns.name = None

# Ordenar columnas: FOTO, Clase, luego modelo_conf alfabéticamente
cols_base = ['FOTO', 'Clase']
cols_conf = [c for c in df_pivot.columns if c not in cols_base]
# Ordenar por tamaño y confianza para que quede lógico
def orden_col(c):
    if c in cols_base:
        return ('', 0, 0)
    partes = c.split('_')
    tam = partes[0]
    conf = float(partes[1])
    orden_tam = {'n':0, 's':1, 'm':2, 'l':3, 'x':4}
    return ('', orden_tam.get(tam, 99), conf)

cols_conf = sorted(cols_conf, key=orden_col)
df_final = df_pivot[cols_base + cols_conf]

print(f'Filas: {len(df_final)} | Columnas: {len(df_final.columns)}')
print('\n📋 Vista previa:')
print(df_final.head(10).to_string())

In [ ]:
# ============================================================
# 6. GUARDAR Y DESCARGAR
# ============================================================

df_final.to_excel(RUTA_SALIDA, index=False)
print(f'💾 Guardado en: {RUTA_SALIDA}')

from google.colab import files
files.download(str(RUTA_SALIDA))